# {title}

状态：**{status}**。单句生成用于体验控制方式，不代表标签准确率或稳定性能。

[官方代码]({source_url}) · [模型权重]({model_url}) · 许可：{license}。

本例使用参考音频控制音色、8 维向量控制情绪。请仅上传有权使用的参考声音。


## 1. 检查环境
官方要求 Python 3.10–3.11；当前 Colab 默认 Python 3.13，需使用受支持的 Python 3.11 本地运行时连接 Colab。该运行时还应挂载 `/mnt/e/models`。

In [ ]:
import platform, shutil, subprocess, sys
from pathlib import Path
assert Path('/mnt/e/models').exists(), '请连接可访问 /mnt/e/models 的 Python 3.11 GPU 本地运行时'
assert shutil.which('nvidia-smi'), '请先选择 GPU 运行时'
print(subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'], text=True))
print('Python:', sys.version)
print('Platform:', platform.platform())
print('空闲磁盘 GiB:', round(shutil.disk_usage('/mnt/e/models').free / 2**30, 2))
assert sys.version_info[:2] in ((3, 10), (3, 11)), 'IndexTTS-2.5 官方依赖仅支持 Python 3.10–3.11；请切换 Python 3.11 本地运行时'
assert shutil.disk_usage('/mnt/e/models').free / 2**30 >= 15


## 2. 安装官方源码
官方推荐 `uv sync`，以下用同一 pyproject 安装最小推理依赖。若依赖解析失败，请保留错误记录。

In [ ]:
SOURCE_REVISION = 'd9e41aac89fd00b3d71497fddb287b7f24613712'
SOURCE_DIR = Path('/tmp/index-tts')
if not SOURCE_DIR.exists():
    subprocess.run(['git', 'clone', 'https://github.com/index-tts/index-tts.git', str(SOURCE_DIR)], check=True)
subprocess.run(['git', '-C', str(SOURCE_DIR), 'fetch', 'origin', SOURCE_REVISION], check=True)
subprocess.run(['git', '-C', str(SOURCE_DIR), 'checkout', '--detach', SOURCE_REVISION], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(SOURCE_DIR), 'huggingface_hub'], check=True)
print('源码:', SOURCE_REVISION)


## 3. 下载固定模型
权重下载到 `/mnt/e/models/indextts2_5`；仓库和权重适用 bilibili Model Use License Agreement。

In [ ]:
import os
os.environ['HF_HOME'] = '/mnt/e/models/.hf-cache'
from huggingface_hub import snapshot_download
MODEL_DIR = Path('/mnt/e/models/indextts2_5')
MODEL_DIR.mkdir(parents=True, exist_ok=True)
MODEL_REVISION = '{model_revision}'
snapshot_download(repo_id='{model_name}', revision=MODEL_REVISION, local_dir=str(MODEL_DIR))
print('模型:', MODEL_DIR, MODEL_REVISION)
import hashlib
for name in ('gpt.pth', 'codec.pth', 's2mel.pth'):
    path = MODEL_DIR / name
    if path.exists():
        with path.open('rb') as f:
            digest = hashlib.file_digest(f, 'sha256').hexdigest()
        print('SHA-256:', name, path.stat().st_size, digest)
import importlib.metadata as metadata
import torch
from indextts.infer_v2_5 import IndexTTS2
for pkg in ('indextts', 'torch', 'transformers', 'huggingface_hub'):
    try:
        print(pkg, metadata.version(pkg))
    except metadata.PackageNotFoundError:
        print(pkg, '版本不可用')
print('CUDA:', torch.version.cuda, torch.cuda.get_device_name(0))
tts = IndexTTS2(cfg_path=str(MODEL_DIR / 'config.yaml'), model_dir=str(MODEL_DIR), use_bf16=False, use_cuda_kernel=False)


## 4. 上传音色参考音频
上传一条有权使用的短音频。

In [ ]:
from google.colab import files
from IPython.display import Audio, display
uploaded = files.upload()
assert len(uploaded) == 1, '请只上传一条参考音频'
name, data = next(iter(uploaded.items()))
REFERENCE_PATH = Path('/tmp') / Path(name).name
REFERENCE_PATH.write_bytes(data)
print('参考音频:', REFERENCE_PATH, '字节:', len(data))
display(Audio(filename=str(REFERENCE_PATH)))


## 5. 输入文本和情绪向量
向量顺序：happy、angry、sad、afraid、disgusted、melancholic、surprised、calm。计时包含官方 `infer` 调用，不含加载、上传、写盘后的播放。

In [ ]:
import time, soundfile as sf
TEXT = '今天的阳光很好，我们一起去散步吧。'
EMOTION_VECTOR = [0.8, 0, 0, 0, 0, 0, 0, 0]
assert len(EMOTION_VECTOR) == 8 and all(0 <= x <= 1.2 for x in EMOTION_VECTOR)
OUTPUT_PATH = Path('/tmp/indextts2_5_emotion.wav')
torch.cuda.synchronize()
torch.cuda.reset_peak_memory_stats()
start = time.perf_counter()
tts.infer(spk_audio_prompt=str(REFERENCE_PATH), text=TEXT, lang='ZH', output_path=str(OUTPUT_PATH), emo_vector=EMOTION_VECTOR, emo_alpha=0.8, use_random=False, duration_factor=1.0, verbose=False)
torch.cuda.synchronize()
elapsed = time.perf_counter() - start
wav, sample_rate = sf.read(str(OUTPUT_PATH))
print('文本:', TEXT, '情绪向量:', EMOTION_VECTOR)
print('采样率:', sample_rate, '音频时长 s:', round(len(wav) / sample_rate, 2))
print('生成耗时 s:', round(elapsed, 3))
print('GPU 峰值 allocated MiB:', round(torch.cuda.max_memory_allocated() / 2**20, 1))
print('GPU 峰值 reserved MiB:', round(torch.cuda.max_memory_reserved() / 2**20, 1))


In [ ]:
from IPython.display import Audio, display
from google.colab import files
print('输出:', OUTPUT_PATH, '字节:', OUTPUT_PATH.stat().st_size)
display(Audio(filename=str(OUTPUT_PATH)))
files.download(str(OUTPUT_PATH))


## 结果边界
状态：**{status}**。向量是输入条件，输出情绪仍需独立核验；参考音频须有使用许可。